# ToolShortlist: train one config on a Kaggle GPU

One notebook run = one training config and one seed. It clones the repository, mines hard negatives, trains, evaluates the checkpoint on every test set, and leaves everything to download under `/kaggle/working/outputs`.

Before running:

1. **Settings** (right-hand panel): Accelerator **GPU T4 x2** (one GPU is used), **Internet on** (the data and base models are downloaded from the Hugging Face Hub).
2. Set `RUN`, `SEED` and `REPO_URL` in the next cell. `RUN` is a file name in `configs/train/` without `.toml`.
3. **Run All**. A `default` run takes about 30 minutes; `base-bge-base` about an hour.

What to bring back (download from the Output tab, or `kaggle kernels output`):

- `outputs/results/train-<RUN>-s<SEED>.json` goes into the repository's `results/` directory. This is the only file the tables need.
- `outputs/runs/train-<RUN>-s<SEED>/summary.json` and `log.jsonl` go into the repository under `runs/` (ignored by git; keep them for the write-up).
- `outputs/runs/train-<RUN>-s<SEED>/model/` is the checkpoint. Keep the `default` seed-0 one for Stage 4; the rest are optional.

In [ ]:
RUN = "default"       # default | text-name | text-name-description | no-hard-negatives | base-minilm | base-bge-base
SEED = 0              # the default config is run with seeds 0, 1 and 2; ablations with seed 0
REPO_URL = "https://github.com/OWNER/toolshortlist.git"   # the public repository
BRANCH = "main"

In [ ]:
import os, subprocess, sys
if not os.path.exists("/kaggle/working/toolshortlist"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "/kaggle/working/toolshortlist"], check=True)
os.chdir("/kaggle/working/toolshortlist")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bm25s>=0.2"], check=True)
subprocess.run(["git", "log", "-1", "--format=commit %H %cd"], check=True)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)
import torch, transformers; print("torch", torch.__version__, "transformers", transformers.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# Train. Hard negatives are mined first (about 10 minutes on a T4); the log prints the loss every 50 steps
# and the validation score every 500.
subprocess.run([sys.executable, "-m", "toolshortlist.train", f"configs/train/{RUN}.toml",
                "--seed", str(SEED), "--out", "/kaggle/working/outputs/runs"], check=True)

In [ ]:
# Evaluate the checkpoint on every test set, exactly as the Stage 1 baselines were evaluated.
import shutil
os.makedirs("/kaggle/working/outputs/results", exist_ok=True)
shutil.copy("results/data_audit.json", "/kaggle/working/outputs/results/")  # decides seen- and unseen-source tasks
run_name = f"train-{RUN}-s{SEED}"
subprocess.run([sys.executable, "-m", "toolshortlist.evaluate", f"/kaggle/working/outputs/runs/{run_name}/eval.toml",
                "--out-dir", "/kaggle/working/outputs/results"], check=True)

In [ ]:
import json
result = json.load(open(f"/kaggle/working/outputs/results/{run_name}.json"))
summary = json.load(open(f"/kaggle/working/outputs/runs/{run_name}/summary.json"))
print("best validation:", summary["best"]["validation"], "at step", summary["best"]["step"], "of", summary["steps"])
toolret = result["benchmarks"]["toolret"]["settings"]["query"]["splits"]
for part in ("unseen_source", "seen_source"):
    print(part, {m: round(100 * v, 2) for m, v in toolret[part]["query_mean"].items()})
live = result["benchmarks"]["livemcpbench"]["settings"]["query"]["all"]
print("livemcpbench", {m: round(100 * v, 2) for m, v in live["query_mean"].items()})
print()
print("Download these from /kaggle/working/outputs:")
for path in (f"results/{run_name}.json", f"runs/{run_name}/summary.json", f"runs/{run_name}/log.jsonl", f"runs/{run_name}/model/"):
    print("  ", path)